In [4]:
#!uv add -U langchain-core langchain-classic langchain-community

In [ ]:
#!uv add -U langchain-openai langchain-chroma langchain-text-splitters

In [ ]:
#!uv add -U rank-bm25 faiss-cpu lark

In [8]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pathlib import Path

data_dir = Path("data/retriever")

documents = [
    Document(
        page_content=file_path.read_text(encoding='utf-8'),
        metadata={
            "source": file_path.name,
            "path": str(file_path.relative_to(data_dir)),
        },
    )
    for file_path in sorted(data_dir.glob("*"))
    if file_path.is_file()
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4.1-mini")
vector_store = InMemoryVectorStore.from_documents(documents, embeddings)

In [9]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":2, "fetch_k":4, "lambda_mult":0.7},
)

results=retriever.invoke("모터 이상 발생 시 점검 항목")
for document in results:
    print(document.metadata["source"], document.page_content)

11_compression_long_report.md # 모터 설비 월간 점검 보고서

## 생산 현황
이번 달 생산량은 계획 대비 98%였고 설비 가동률은 91%였습니다. 주간 생산 계획에는 큰 차이가 없었습니다.

## 청소 상태
설비 외관 청소 상태는 양호했습니다. 모터 주변 통로에 자재가 일부 적치되어 이동 조치했습니다.

## 진동 이상
9월 18일 오전부터 모터 진동이 평균 2.2 mm/s에서 6.1 mm/s까지 증가했습니다.
점검 결과 베어링 윤활 상태가 부족했고 축 정렬 오차도 발견되었습니다.
윤활 보충과 축 정렬 작업 후 진동은 2.5 mm/s 수준으로 감소했습니다.

## 온도
진동 증가 시간대에 베어링 온도도 58도에서 76도까지 상승했습니다.
조치 후 온도는 61도로 회복되었습니다.

## 전력
모터 전력 사용량은 전월 대비 2% 증가했으나 생산량 차이를 고려하면 유의한 변화는 아니었습니다.

## 안전
월간 안전 점검에서 비상 정지 버튼과 안전 커버의 이상은 발견되지 않았습니다.

## 향후 조치
진동과 온도가 동시에 증가할 경우 우선 점검 알람을 발생하도록 기준을 검토합니다.

09_timeweighted_documents.json [
  {
    "document_id": "T-001",
    "text": "모터 진동 증가 시 베어링 마모와 축 정렬을 점검합니다.",
    "event_time": "2026-09-01T09:00:00+09:00",
    "topic": "motor",
    "importance": "high"
  },
  {
    "document_id": "T-002",
    "text": "모터 베어링 온도가 75도를 초과해 윤활 상태를 점검했습니다.",
    "event_time": "2026-09-20T14:30:00+09:00",
    "topic": "motor",
    "importance": "high"
  },
  {
    "document_id": "T-003",
    "te

In [22]:
import csv
from pathlib import Path
from langchain_core.documents import Document

data_dir = Path("data/retriever")
target_files = ["01_common_documents.csv","03_keyword_bm25_documents.csv","04_reorder_documents.csv", "07_selfquery_documents.csv"]

documents=[]
for name in target_files:
    file_path = data_dir / name
    with file_path.open(encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            metadata = dict(row)
            metadata.pop("text", None)
            for key in ("year", "priority", "original_order"):
                if key in metadata:
                    metadata[key] = int(metadata[key])
            metadata["source"] = file_path.name
            documents.append(
                Document(
                    page_content=row["text"],
                    metadata=metadata
                )
            )

In [23]:
print(len(documents))
print(documents[0].metadata)
print(sum(1 for d in documents if d.metadata.get("equipment") == "motor"))

vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
filtered_retriever = vector_store.as_retriever(
    search_kwargs={"k":2, "filter": lambda doc: doc.metadata.get("equipment") =="motor"},
)

results=filtered_retriever.invoke("모터 최근 점검 결과")

36
{'document_id': 'M-001', 'equipment': 'press', 'year': 2025, 'priority': 3, 'category': 'maintenance', 'plant': 'plant_a', 'source': '01_common_documents.csv'}
5


In [24]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

base_retriever = vector_store.as_retriever(search_kwargs={"k":4})
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor,
)

results = compression_retriever.invoke("모터 진동 원인과 조치")
for document in results:
    print(document.page_content)
    print(document.metadata)
    print("---")

모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.
{'document_id': 'R-003', 'original_order': 3, 'source': '04_reorder_documents.csv'}
---
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
{'document_id': 'SQ-003', 'equipment': 'motor', 'year': 2025, 'priority': 3, 'category': 'maintenance', 'plant': 'plant_a', 'source': '07_selfquery_documents.csv'}
---
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
{'document_id': 'M-002', 'equipment': 'motor', 'year': 2024, 'priority': 2, 'category': 'maintenance', 'plant': 'plant_a', 'source': '01_common_documents.csv'}
---
모터의 이름판과 정격 전압을 확인합니다.
{'document_id': 'R-004', 'original_order': 4, 'source': '04_reorder_documents.csv'}
---


In [25]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3
vector_retriever = vector_store.as_retriever(search_kwargs={"k":3})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

results = ensemble_retriever.invoke("M-002 모터 축 정렬")
for document in results:
    print(document.metadata["document_id"], "|", document.metadata["source"])
    print(document.page_content)
    print("---")

M-002 | 01_common_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
---
R-003 | 04_reorder_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.
---


In [30]:
from langchain_community.document_transformers import LongContextReorder

retriever = vector_store.as_retriever(search_kwargs={"k":4})
retrieved_documents = retriever.invoke("모터 점검과 이상 조치")

reordering = LongContextReorder()
reordered_documents = reordering.transform_documents(retrieved_documents)

for index, document in enumerate(reordered_documents, start=1):
    print(index, document.metadata["document_id"], "|", document.metadata["source"])
    print(document.page_content)

context = "\n\n".join(
    document.page_content for document in reordered_documents
)

1 SQ-003 | 07_selfquery_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
2 R-004 | 04_reorder_documents.csv
모터의 이름판과 정격 전압을 확인합니다.
3 M-002 | 01_common_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬 상태를 점검합니다.
4 R-003 | 04_reorder_documents.csv
모터 진동 증가 시 베어링 마모와 축 정렬을 우선 점검합니다.
